# 03 ML - Taxi Fare Prediction with Spark MLlib

This notebook trains a scalable baseline regression model on curated taxi trip data and persists both prediction outputs and the serialized model to storage.

## Imports and Spark session

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

from src.ml_pipeline import train_fare_model
from src.paths import ML_PATH, MODEL_PATH, SILVER_PATH
from src.transform import select_model_columns

builder = (
    SparkSession.builder.appName("nyc-taxi-ml")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Reading model inputs from: {SILVER_PATH}")
print(f"Prediction output path: {ML_PATH}")
print(f"Model artifact path: {MODEL_PATH}")

## Load training dataset

In [ ]:
silver_df = spark.read.format("delta").load(SILVER_PATH)
model_df = select_model_columns(silver_df)

model_df.printSchema()
model_df.show(5, truncate=False)

## Train model and evaluate RMSE

In [ ]:
model, predictions_df, rmse = train_fare_model(model_df)

print(f"Model RMSE: {rmse:.4f}")
predictions_df.select(
    "fare_amount",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "hour",
    "prediction",
).show(20, truncate=False)

## Persist predictions and trained model

In [ ]:
(
    predictions_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(ML_PATH)
)

model.write().overwrite().save(MODEL_PATH)

print("Predictions and model artifacts saved successfully.")